## Part 1: 安装和配置

In [ ]:
# 安装依赖
!pip install agentlightning openai aiohttp python-dotenv

In [ ]:
import os
import asyncio
import agentlightning as agl

# 配置 OpenAI API（或替换为本地 vLLM）
os.environ["OPENAI_API_KEY"] = "sk-your-key"  # 替换成你的Key
os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"

print("✅ 环境配置完成")

## Part 2: 基础追踪 - 记录 Agent 行为

这个例子来自 `examples/minimal/write_traces.py`，演示如何用 `OtelTracer` 记录轨迹。

In [ ]:
async def basic_tracing_example():
    """演示最基础的追踪功能（不调用模型）"""
    
    # 1. 创建追踪器和存储
    tracer = agl.OtelTracer()
    store = agl.InMemoryLightningStore()
    
    # 2. 开始一个 Rollout（代表一次任务执行）
    rollout = await store.start_rollout(input={"task": "计算 2+2"})
    
    # 3. 在 tracer 的生命周期内记录行为
    with tracer.lifespan(store):
        async with tracer.trace_context(
            "demo-trace",
            store=store,
            rollout_id=rollout.rollout_id,
            attempt_id=rollout.attempt.attempt_id
        ):
            # 记录第一步：思考
            with tracer.start_as_current_span("step-1-thinking"):
                print("🤔 Agent 正在思考...")
            
            # 记录第二步：执行
            with tracer.start_as_current_span("step-2-execution"):
                result = 2 + 2
                print(f"💡 Agent 计算结果: {result}")
            
            # 记录奖励
            agl.emit_reward(1.0 if result == 4 else 0.0)
            print("🏆 奖励: 1.0")
    
    # 4. 查询记录的轨迹
    spans = await store.query_spans(rollout_id=rollout.rollout_id)
    print(f"\n📊 共记录了 {len(spans)} 个 Span")
    for span in spans:
        print(f"  ▸ {span.name}")

# 运行
await basic_tracing_example()

## Part 3: 真实的 LLM 调用 - 使用 OpenAI API

这个例子基于 `examples/minimal/llm_proxy.py`，展示如何通过 Agent Lightning 调用真实模型。

In [ ]:
import aiohttp

async def llm_call_with_tracing():
    """使用 LLM Proxy 调用模型并自动追踪"""
    
    # 1. 创建 Store Server（负责管理追踪数据）
    store = agl.InMemoryLightningStore()
    store_server = agl.LightningStoreServer(store, "127.0.0.1", 43887)
    await store_server.start()
    print("✅ Store Server 已启动")
    
    # 2. 创建 LLM Proxy（代理 OpenAI API，自动追踪）
    llm_proxy = agl.LLMProxy(
        port=43886,
        model_list=[
            {
                "model_name": "gpt-4o-mini",
                "litellm_params": {
                    "model": "openai/gpt-4o-mini",
                },
            }
        ],
        store=store_server,
        callbacks=["opentelemetry"],
    )
    await llm_proxy.start()
    print("✅ LLM Proxy 已启动在 http://localhost:43886")
    
    try:
        # 3. 开始一个 Rollout
        store_client = agl.LightningStoreClient("http://localhost:43887")
        rollout = await store_client.start_rollout(input={"query": "What is AI?"})
        
        # 4. 通过 Proxy 调用 LLM（自动追踪）
        chat_url = (
            f"http://localhost:43886/rollout/{rollout.rollout_id}/"
            f"attempt/{rollout.attempt.attempt_id}/v1/chat/completions"
        )
        
        async with aiohttp.ClientSession() as session:
            async with session.post(
                chat_url,
                json={
                    "model": "gpt-4o-mini",
                    "messages": [
                        {"role": "user", "content": "What is AI? Answer in one sentence."}
                    ],
                },
            ) as response:
                result = await response.json()
                answer = result["choices"][0]["message"]["content"]
                print(f"\n🤖 模型回答: {answer}")
        
        # 5. 查询追踪到的 Spans
        spans = await store_client.query_spans(
            rollout_id=rollout.rollout_id,
            attempt_id=rollout.attempt.attempt_id
        )
        print(f"\n📊 共追踪到 {len(spans)} 个 Span")
        for span in spans:
            print(f"  ▸ {span.name}")
            if "gen_ai" in span.name:
                print(f"    Model: {span.attributes.get('gen_ai.request.model')}")
        
        await store_client.close()
        
    finally:
        # 6. 清理资源
        await llm_proxy.stop()
        await store_server.stop()
        print("\n✅ 服务已停止")

# 运行（需要有效的 OpenAI API Key）
# await llm_call_with_tracing()

**说明**：
- `LLMProxy` 会拦截所有发往 LLM 的请求，自动记录 Prompt、Response、Token 使用量等
- URL 中包含 `rollout_id` 和 `attempt_id`，确保追踪数据归档正确
- 这就是 Agent Lightning 的核心：**无感追踪，零侵入**

## Part 4: 定义一个可训练的 Agent

这个例子简化自 `examples/calc_x/calc_agent.py`，展示如何定义一个数学 Agent。

In [ ]:
from typing import TypedDict
import re

# 1. 定义任务结构
class MathTask(TypedDict):
    question: str
    answer: str

# 2. 使用 @agl.rollout 装饰器定义 Agent
@agl.rollout
async def simple_math_agent(task: MathTask, llm: agl.LLM) -> None:
    """
    一个简单的数学 Agent，通过 LLM 解题。
    
    Args:
        task: 包含问题和答案的任务
        llm: Agent Lightning 提供的 LLM 资源（包含端点、模型名、温度等）
    
    这个函数会被自动追踪，无需手动添加 tracing 代码。
    """
    print(f"📝 问题: {task['question']}")
    
    # 在真实场景中，你会通过 llm.endpoint 调用模型
    # 这里简化为直接调用 OpenAI（需要 openai 包）
    from openai import AsyncOpenAI
    
    client = AsyncOpenAI(
        base_url=llm.endpoint,
        api_key=os.getenv("OPENAI_API_KEY", "dummy")
    )
    
    response = await client.chat.completions.create(
        model=llm.model,
        messages=[
            {"role": "user", "content": f"{task['question']} Output only the number."}
        ],
        temperature=llm.sampling_parameters.get("temperature", 0.0),
    )
    
    agent_answer = response.choices[0].message.content.strip()
    print(f"🤖 Agent 答案: {agent_answer}")
    print(f"✅ 正确答案: {task['answer']}")
    
    # 3. 计算奖励
    # 简单判断：答案是否匹配
    reward = 1.0 if agent_answer == task['answer'] else 0.0
    
    # 4. 发出奖励信号（这会被 Agent Lightning 自动记录）
    agl.emit_reward(reward)
    print(f"🏆 奖励: {reward}\n")

print("✅ Agent 定义完成")

## Part 5: 手动运行 Agent（调试模式）

在训练之前，先手动运行 Agent 测试效果。

In [ ]:
async def debug_run_agent():
    """手动运行 Agent 进行调试"""
    
    # 1. 创建追踪器和存储
    tracer = agl.OtelTracer()
    store = agl.InMemoryLightningStore()
    
    # 2. 创建 Runner
    runner = agl.LitAgentRunner[MathTask](tracer)
    
    # 3. 定义 LLM 资源
    llm_resource = agl.LLM(
        endpoint=os.getenv("OPENAI_BASE_URL"),
        model="gpt-4o-mini",
        sampling_parameters={"temperature": 0.0}
    )
    
    # 4. 准备测试任务
    test_tasks = [
        MathTask(question="What is 12 + 8?", answer="20"),
        MathTask(question="What is 15 * 3?", answer="45"),
        MathTask(question="What is 100 / 5?", answer="20"),
    ]
    
    # 5. 运行 Agent
    print("🚀 开始运行 Agent...\n")
    with runner.run_context(agent=simple_math_agent, store=store):
        for i, task in enumerate(test_tasks, 1):
            print(f"=== 任务 {i}/{len(test_tasks)} ===")
            await runner.step(
                task,
                resources={"main_llm": llm_resource}
            )
    
    print("✅ 所有任务完成！")

# 运行（需要 OpenAI API Key）
# await debug_run_agent()

## Part 6: 理解训练流程

真实的训练代码在 `examples/calc_x/train_calc_agent.py`，这里展示其核心逻辑：

```python
# 伪代码：实际训练流程

# 1. 启动 Ray 集群（分布式计算）
ray.init()

# 2. 加载数据
train_data = pd.read_parquet("data/train.parquet")

# 3. 创建 Trainer（包含 Actor, Critic, Reference 三个模型）
trainer = agl.verl.AgentLightningTrainer(
    store=store,
    llm_proxy=llm_proxy,
    config={
        "model_path": "Qwen/Qwen2.5-7B-Instruct",  # 你要训练的开源模型
        "algorithm": "PPO",
        "learning_rate": 1e-5,
        # ... 更多配置
    }
)

# 4. 训练循环
for epoch in range(10):
    for batch in train_data:
        # 4.1 Rollout: Actor 模型运行 Agent，产生轨迹
        rollouts = trainer.generate_rollouts(batch)
        
        # 4.2 Reward: 评估每个轨迹的得分
        rewards = [r.final_reward for r in rollouts]
        
        # 4.3 Advantage: Critic 预测价值，计算优势
        advantages = trainer.compute_advantages(rollouts)
        
        # 4.4 Update: 更新 Actor 和 Critic 的参数
        trainer.update_models(rollouts, advantages)
        
        # 4.5 KL Penalty: 用 Reference 模型约束，防止跑偏
        kl_divergence = trainer.compute_kl(rollouts)
        
        print(f"Epoch {epoch}, Avg Reward: {np.mean(rewards):.2f}, KL: {kl_divergence:.4f}")

# 5. 保存训练好的模型
trainer.save("./checkpoints/my_trained_agent")
```

## 总结

### Agent Lightning 的三大核心

1. **Tracer (追踪器)**
   - 自动记录 Agent 的所有行为
   - 无需修改 Agent 代码

2. **LLM Proxy (模型代理)**
   - 拦截所有 LLM 调用
   - 自动记录 Prompt、Response、Token IDs

3. **VERL Trainer (训练器)**
   - 管理 Actor、Critic、Reference 三个模型
   - 实现完整的 PPO 训练循环

### 使用场景

✅ **适合用 Agent Lightning**：
- 训练会使用工具的 Agent（代码生成、API 调用）
- 训练多步推理的 Agent（数学、逻辑）
- 训练与环境交互的 Agent（游戏、机器人）

❌ **不适合**：
- 单纯的对话模型（用 SFT 就够了）
- 训练闭源模型（拿不到权重）
- 单轮问答任务（不需要 RL）

### 下一步

1. 查看完整示例：`examples/calc_x/` - 数学 Agent 训练
2. 阅读文档：https://microsoft.github.io/agent-lightning/
3. 准备数据集和 GPU，开始真正的训练

**关键资源需求**：
- 至少 1 张 40GB GPU（A100）
- 开源模型权重（Qwen/Llama/Mistral）
- 带奖励标注的数据集